# ARCH 6131 Scale | 分步图纸与筛选
按顺序运行。每节的 **输入** 写在代码上方，代码运行后显示字段预览、统计和图。每张图单独保存 PNG 和 PDF。读取既有综合成果，复用上次的五层筛选。各主题图和各层筛选图分别出图。所有地块填色按研究边界裁切，统计与筛选沿用已保存的原地块指标。

- 第 0 节建立底图与公共设置。
- 第 1–7 节阅读各指标。百分比与数量均保留原统计口径。
- 第 8–12 节逐层筛选。每张图用同一比例尺和范围，展示该层留下、排除和待核实的地块。
- 所有文件存入 Drive 的 `ARCH6131_Scale/outputs/step_figures/<运行时间>/`。


## 0｜读取底图
**输入** `outputs/all_enriched_plots.geojson` 的地块 ID 与几何，以及 `data/raw/boundary.geojson` 的研究范围。  
**输出** 数据量、字段检查、研究范围与初始候选地块图。后续地图统一使用 EPSG:2263。


In [ ]:
%pip -q install geopandas matplotlib

from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime
from pathlib import Path
import textwrap
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator
from IPython.display import display

ROOT = Path("/content/drive/MyDrive/ARCH6131_Scale")
RAW, OUT = ROOT / "data/raw", ROOT / "outputs"
CRS = 2263

def read_layer(path):
    if not path.exists():
        raise FileNotFoundError(f"找不到 {path}。请检查 Drive 里已保存的成果。")
    layer = gpd.read_file(path)
    if layer.empty or layer.crs is None:
        raise ValueError(f"{path.name} 没有数据或缺少坐标系")
    return layer.to_crs(CRS)

plots_path = OUT / "all_enriched_plots.geojson"
plots = read_layer(plots_path)
boundary = read_layer(RAW / "boundary.geojson")
if plots.plot_id.isna().any() or not plots.plot_id.is_unique:
    raise ValueError("地块编号有缺失或重复")
plots["plot_id"] = plots.plot_id.astype(str)

# 缓存显示用的几何交集，后续仍用原始 plots 计算指标和筛选。
study_mask = boundary.geometry.union_all()
display_plots = gpd.clip(plots[["plot_id", "geometry"]], study_mask, keep_geom_type=True)
display_plots = display_plots.loc[~display_plots.geometry.is_empty].copy()
if display_plots.empty:
    raise ValueError("研究边界内没有可绘制的地块面积")
display_geometry = display_plots.set_index("plot_id").geometry

def plots_for_display(layer):
    visible = layer.loc[layer.plot_id.isin(display_geometry.index)].copy()
    return visible.set_geometry(visible.plot_id.map(display_geometry), crs=CRS)

run_dir = OUT / "step_figures" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
run_dir.mkdir(parents=True)
paper, ink = "#faf8f3", "#173447"
land, line = "#e3e5df", "#829096"
orange, teal = "#d95331", "#137e72"
xmin, ymin, xmax, ymax = boundary.total_bounds
width, height = xmax - xmin, ymax - ymin
xlimits = (xmin - .04 * width, xmax + .04 * width)
ylimits = (ymin - .06 * height, ymax + .06 * height)
plt.rcParams.update({"font.family": "DejaVu Sans", "pdf.fonttype": 42})

def start_map(number, title, subtitle):
    fig, ax = plt.subplots(figsize=(11, 8), facecolor=paper)
    fig.subplots_adjust(left=.08, right=.92, top=.83, bottom=.15)
    fig.text(.08, .94, f"{number}   {title}", fontsize=22, color=ink, weight="bold")
    fig.text(.08, .89, textwrap.fill(subtitle, 112), fontsize=10, color=ink)
    ax.set_facecolor(paper)
    return fig, ax

def draw_base(ax):
    display_plots.plot(ax=ax, color=land, edgecolor="#c7ceca", linewidth=.12, rasterized=True)

def finish_map(ax):
    boundary.boundary.plot(ax=ax, color=line, linewidth=.7)
    ax.set_xlim(*xlimits)
    ax.set_ylim(*ylimits)
    ax.set_aspect("equal")
    ax.set_axis_off()
    x, y = xlimits[0] + .025 * width, ylimits[0] + .03 * height
    ax.plot([x, x + 1000], [y, y], color=ink, linewidth=2)
    ax.text(x, y + .012 * height, "1,000 ft", fontsize=8, color=ink)
    ax.annotate("Grid N", xy=(.97, .96), xytext=(.97, .85),
                xycoords="axes fraction", ha="center", fontsize=8,
                arrowprops={"arrowstyle": "->", "color": ink})

def save_figure(fig, filename, note):
    fig.text(.08, .052, textwrap.fill(note, 123), fontsize=9, color=ink)
    for ext in ("png", "pdf"):
        target = run_dir / f"{filename}.{ext}"
        fig.savefig(target, dpi=220, facecolor=paper)
        print("输出", target.name)
    plt.show()
    plt.close(fig)

print("输入", plots_path.name, "和 boundary.geojson")
print("初始地块", f"{len(plots):,}")
print("输出目录", run_dir)
display(plots[["plot_id", "LandUse", "BldgClass"]].head())
fig, ax = start_map("00", "Study area and baseline plots",
                    "Jackson Heights, North Corona and Corona")
draw_base(ax)
finish_map(ax)
save_figure(fig, "00_study_area", "Source: NYC MapPLUTO and NYC NTA study boundary.")


## 专题图的公共函数
下面的函数先列出输入字段和数值摘要，再绘制单张地图。同一函数在各节中使用不同的数据，灰色表示缺失或无匹配覆盖。  
**年龄和健康**以 Census Tract 为单位。**建筑、食品店和事故**以地块为单位。


In [ ]:
def draw_metric(number, filename, layer, field, title, subtitle, unit, palette, source_name, note):
    print("输入", source_name, "·", field)
    ids = [c for c in ("plot_id", "GEOID", "geoid") if c in layer.columns]
    values = pd.to_numeric(layer[field], errors="coerce")
    print(f"总记录 {len(layer):,} · 有值 {values.notna().sum():,} · 缺失 {values.isna().sum():,}")
    display(layer[ids + [field]].head())
    display(values.describe().to_frame(unit))
    if values.notna().sum() == 0:
        raise ValueError(f"{field} 没有可绘制的数值")
    fig, ax = start_map(number, title, subtitle)
    draw_base(ax)
    mapped = layer.assign(_value=values)
    if "plot_id" in mapped.columns:
        mapped = plots_for_display(mapped)
    missing = mapped.loc[mapped["_value"].isna()]
    if not missing.empty:
        missing.plot(ax=ax, color="#c0c4c2", edgecolor=line,
                     linewidth=.2, rasterized=True)
    mapped_values = mapped.loc[mapped["_value"].notna()]
    if mapped_values.empty:
        raise ValueError(f"{field} 在研究边界内没有可绘制的数值")
    mapped_values.plot(
        ax=ax, column="_value", cmap=palette, legend=True,
        edgecolor="none", rasterized=True,
        legend_kwds={"orientation": "horizontal", "label": unit,
                     "shrink": .58, "pad": .018, "aspect": 35},
    )
    finish_map(ax)
    save_figure(fig, filename, note)

def study_tracts(filename):
    layer = read_layer(RAW / filename)
    return gpd.clip(layer, study_mask, keep_geom_type=True)


## 01｜建筑覆盖
**输入** `all_enriched_plots.geojson` 中的 `building_coverage_pct`。地块边界与建筑轮廓交集的面积比例。
**输出** 字段预览、数值摘要与独立地图。


In [ ]:
draw_metric(
    "01", "01_building_coverage", plots, "building_coverage_pct",
    "Building coverage",
    "Share of each plot covered by mapped building footprints", "Coverage (%)", "YlGnBu",
    "all_enriched_plots.geojson",
    "Sources: MapPLUTO and NYC Building Footprints. Footprint overlap is counted once.",
)


## 02｜食品零售
**输入** `all_enriched_plots.geojson` 中的 `food_store_count_buffer`。每个地块边界外 660 ft 内有坐标的零售许可证数量。
**输出** 字段预览、数值摘要与独立地图。


In [ ]:
draw_metric(
    "02", "02_food_retail", plots, "food_store_count_buffer",
    "Nearby food retail",
    "Mapped retail licences within 660 ft of each plot", "Retail licences (count)", "YlOrBr",
    "all_enriched_plots.geojson",
    "Source: NYS Retail Food Stores. Values reflect mapped licences and straight-line proximity.",
)


## 03｜交通事故总数
**输入** `all_enriched_plots.geojson` 中的 `crash_count_buffer`。每个地块边界外 660 ft 内的全部记录事故。
**输出** 字段预览、数值摘要与独立地图。


In [ ]:
draw_metric(
    "03", "03_all_crashes", plots, "crash_count_buffer",
    "Reported traffic crashes",
    "Crashes within 660 ft of each plot, 2021–2025", "Crash events (count)", "OrRd",
    "all_enriched_plots.geojson",
    "Source: NYPD Motor Vehicle Collisions. The same event can appear in several plot buffers.",
)


## 04｜行人伤亡事故
**输入** `all_enriched_plots.geojson` 中的 `ped_casualty_crash_count_buffer`。同一缓冲区内有行人受伤或死亡的事故起数。
**输出** 字段预览、数值摘要与独立地图。


In [ ]:
draw_metric(
    "04", "04_pedestrian_casualty_crashes", plots, "ped_casualty_crash_count_buffer",
    "Pedestrian casualty crashes",
    "Crashes involving pedestrian injury or death, 2021–2025", "Crash events (count)", "PuRd",
    "all_enriched_plots.geojson",
    "Source: NYPD Motor Vehicle Collisions. Five-year counts within 660 ft. Exposure is unmeasured.",
)


## 05｜老年人口比例
**输入** `age_tracts.geojson` 中的 `age65_pct`。统计区的 65 岁及以上人口比例，图中仅裁剪显示范围。
**输出** 字段预览、数值摘要与独立地图。


In [ ]:
draw_metric(
    "05", "05_older_adult_share", study_tracts("age_tracts.geojson"), "age65_pct",
    "Residents aged 65 and over",
    "Share of residents by census tract, ACS 2020–2024", "Tract population aged 65+ (%)", "BuPu",
    "age_tracts.geojson",
    "Source: US Census Bureau, ACS 2024 five-year B01001 estimates. Grey indicates missing estimates or tract coverage.",
)


## 06｜成人当前哮喘
**输入** `health_tracts.geojson` 中的 `adult_asthma_pct`。按统计区显示的成人哮喘粗患病率估计。
**输出** 字段预览、数值摘要与独立地图。


In [ ]:
draw_metric(
    "06", "06_adult_asthma", study_tracts("health_tracts.geojson"), "adult_asthma_pct",
    "Adult current asthma",
    "Crude modeled prevalence by census tract", "Adult prevalence (%)", "GnBu",
    "health_tracts.geojson",
    "Source: CDC PLACES. Model-based tract estimates. Grey indicates missing estimates or tract coverage.",
)


## 07｜交通观测
**输入** `outputs/traffic_2024_sites_provisional.geojson` 中的道路、方向、观测日期数与上午 8–9 点平均车辆数。  
**输出** 观测位置图与方向比较条形图，各自导出。位置图把相同坐标的方向合用一个编号，比较图保留方向和样本天数。


In [ ]:
traffic_path = OUT / "traffic_2024_sites_provisional.geojson"
traffic_field = "weekday_08_09_mean_vehicles"

if traffic_path.exists():
    traffic = read_layer(traffic_path)
    traffic[traffic_field] = pd.to_numeric(traffic[traffic_field], errors="coerce")
    traffic = traffic.loc[traffic[traffic_field].notna()].copy()
    if traffic.empty:
        raise ValueError("交通文件没有可绘制的小时车辆数")
    traffic["site_id"] = [
        f"T{i+1}" for i in pd.factorize(traffic.geometry.to_wkb())[0]
    ]
    print("输入", traffic_path.name)
    columns = ["site_id", "street", "fromst", "tost", "direction",
               traffic_field, "observed_dates", "first_date", "last_date"]
    display(traffic[[c for c in columns if c in traffic.columns]])

    fig, ax = start_map("07A", "Traffic observation sites",
                        "2024 weekday observations, 08:00–09:00")
    draw_base(ax)
    points = traffic.drop_duplicates("site_id")
    points.plot(ax=ax, color=teal, markersize=65, edgecolor="white", linewidth=1.2)
    for row in points.itertuples():
        ax.annotate(row.site_id, (row.geometry.x, row.geometry.y), xytext=(6, 5),
                    textcoords="offset points", color=ink, weight="bold")
    finish_map(ax)
    save_figure(fig, "07a_traffic_sites",
                "Source: NYC DOT Automated Traffic Volume Counts. The original point CRS interpretation remains provisional.")

    fig, ax = plt.subplots(figsize=(11, 7), facecolor=paper)
    fig.subplots_adjust(left=.37, right=.91, bottom=.18, top=.78)
    fig.text(.08, .91, "07B   Observed traffic by direction", fontsize=22, color=ink, weight="bold")
    fig.text(.08, .855, "2024 weekdays, 08:00–09:00 | Means across complete observed hours", fontsize=10)
    bars = traffic.sort_values(["site_id", "direction"]).reset_index(drop=True)
    labels = [
        f"{row.site_id}   {row.street}   {row.direction}"
        for row in bars.itertuples()
    ]
    ax.set_facecolor(paper)
    ax.barh(range(len(bars)), bars[traffic_field], color=teal)
    ax.set_yticks(range(len(bars)), labels)
    ax.invert_yaxis()
    ax.set_xlim(0, float(bars[traffic_field].max()) * 1.35 + 1)
    ax.set_xlabel("Mean observed vehicles per hour")
    ax.xaxis.set_major_locator(MaxNLocator(6))
    ax.grid(axis="x", alpha=.2)
    ax.set_axisbelow(True)
    for i, row in bars.iterrows():
        dates = row.get("observed_dates", "?")
        ax.annotate(f"{row[traffic_field]:.1f}  |  {dates} dates",
                    (row[traffic_field], i), xytext=(5, 0),
                    textcoords="offset points", va="center", fontsize=9)
    save_figure(fig, "07b_traffic_directions",
                "Source: NYC DOT. Each direction is shown separately. Observation dates differ across sites.")
else:
    print("未找到交通观测文件，本节跳过。其他图纸仍可生成。")


## 筛选设置｜与上一版综合图一致
**输入** 保存的完整地块指标。所有数值阈值在第一层开始前固定。年龄中位数按初始地块所涉及的不同 Census Tract 计算。  
**输出** 原始候选集合和五条规则，后面的五个单元格按顺序运行。


In [ ]:
rule_fields = [
    "building_coverage_pct", "age65_pct",
    "food_store_count_buffer", "ped_casualty_crash_count_buffer"
]
for field in rule_fields:
    plots[field] = pd.to_numeric(plots[field], errors="coerce")

tract_age = plots[["age65_pct_GEOID", "age65_pct"]].dropna().drop_duplicates()
if not tract_age.age65_pct_GEOID.is_unique:
    raise ValueError("同一统计区有多个年龄比例，请先检查综合成果")

land_use = pd.to_numeric(plots.LandUse, errors="coerce")
building_class = plots.BldgClass.astype("string").str.strip().str.upper()
is_vacant = land_use.eq(11)
is_parking = building_class.isin(["G6", "G7", "Z2", "RP"])
space_eligible = pd.Series(pd.NA, index=plots.index, dtype="boolean")
known = land_use.notna() & building_class.notna() & building_class.ne("")
space_eligible.loc[known] = False
space_eligible.loc[is_vacant | is_parking] = True
plots["space_type_eligible"] = space_eligible
plots["space_type"] = pd.Series(pd.NA, index=plots.index, dtype="string")
plots.loc[is_vacant, "space_type"] = "Vacant land"
plots.loc[is_parking & ~is_vacant, "space_type"] = "Surface parking"

rules = [
    ("Lower building coverage", rule_fields[0], "<=", plots[rule_fields[0]].quantile(.25)),
    ("Higher older-adult share", rule_fields[1], ">=", tract_age.age65_pct.median()),
    ("Fewer nearby food retailers", rule_fields[2], "<=", plots[rule_fields[2]].median()),
    ("More pedestrian casualty crashes", rule_fields[3], ">=", plots[rule_fields[3]].median()),
    ("Vacant land or surface parking", "space_type_eligible", "==", True),
]
if not all(np.isfinite(rule[3]) for rule in rules[:4]):
    raise ValueError("有筛选字段没有有效数值")
display(pd.DataFrame(rules, columns=["筛选层", "字段", "运算", "阈值"]))

# 每一层保留独立结果，重跑某一层时从上一层重新计算。
stage_plots = {0: plots.copy()}

def screen_stage(stage, filename, detail):
    if stage - 1 not in stage_plots:
        raise ValueError("请先运行上一层筛选")
    incoming = stage_plots[stage - 1].copy()
    name, field, op, threshold = rules[stage - 1]
    values = incoming[field]
    unknown = values.isna()
    comparison = {"<=": values.le, ">=": values.ge, "==": values.eq}
    passed = comparison[op](threshold).fillna(False) & ~unknown
    failed = ~unknown & ~passed
    outgoing = incoming.loc[passed].copy()
    if stage == len(rules):
        outgoing = outgoing.sort_values("plot_id")
        outgoing["item_id"] = [f"P{i:02d}" for i in range(1, len(outgoing)+1)]
    # Earlier reruns invalidate downstream results until those cells are rerun.
    for stale in list(stage_plots):
        if stale >= stage:
            del stage_plots[stale]
    stage_plots[stage] = outgoing

    print(f"输入 {len(incoming):,} 个地块 · 字段 {field} · 条件 {op} {threshold}")
    display(incoming[["plot_id", field]].head())
    display(pd.DataFrame([{
        "输入": len(incoming), "保留": int(passed.sum()),
        "本层排除": int(failed.sum()), "本层待核实": int(unknown.sum())
    }]))
    columns = [c for c in ["item_id", "plot_id", field, "space_type"]
               if c in outgoing.columns]
    display(outgoing[columns].head(20))

    number = f"{stage + 7:02d}"
    subtitle = (f"{len(incoming):,} entering  |  {len(outgoing):,} retained  |  "
                f"{int(failed.sum()):,} excluded  |  {int(unknown.sum()):,} for review")
    fig, ax = start_map(number, name, subtitle)
    draw_base(ax)
    groups = [(incoming.loc[failed], "#8e9ca5", "Excluded this step"),
              (incoming.loc[unknown], "#e6ae4e", "Review: missing value")]
    if stage == len(rules):
        groups.extend([
            (outgoing.loc[outgoing.space_type.eq("Vacant land")], orange, "Vacant land"),
            (outgoing.loc[outgoing.space_type.eq("Surface parking")], teal, "Surface parking"),
        ])
    else:
        groups.append((outgoing, teal, "Retained"))
    legend = []
    for group, color, label in groups:
        group = plots_for_display(group)
        if group.empty:
            continue
        group.plot(ax=ax, color=color, edgecolor="none", rasterized=True)
        legend.append(Patch(color=color, label=label))
        if stage == len(rules) and label in {"Vacant land", "Surface parking"}:
            points = group.representative_point()
            ax.scatter(points.x, points.y, color=color, s=22, edgecolor="white", linewidth=.5)
            if len(outgoing) <= 40:
                for item, point in zip(group.item_id, points):
                    ax.annotate(item, (point.x, point.y), xytext=(4,4),
                                textcoords="offset points", fontsize=7, color=ink)
    finish_map(ax)
    if legend:
        ax.legend(handles=legend, loc="upper left", fontsize=8, frameon=True)
    if stage < len(rules):
        criterion = f"Rule: {field} {op} {threshold:.3f}. "
    else:
        criterion = "Rule: PLUTO LandUse 11 or BldgClass G6, G7, Z2 or RP. "
    save_figure(fig, filename, criterion + detail)


## 08｜第 1 层：低建筑覆盖
**输入** 上一层保留的地块与建筑覆盖率。
**输出** 本层计数、地块预览和独立地图。


In [ ]:
screen_stage(1, "08_filter_building",
             "Threshold: 25th percentile of the initial plots. Source: MapPLUTO and NYC Building Footprints.")


## 09｜第 2 层：老年人口比例
**输入** 上一层保留的地块与统计区老年人口比例。
**输出** 本层计数、地块预览和独立地图。


In [ ]:
screen_stage(2, "09_filter_age",
             "Threshold: median of unique tracts represented by the initial plots. Source: US Census Bureau ACS B01001.")


## 10｜第 3 层：周边食品店数量
**输入** 上一层保留的地块与660 ft 内的食品店许可证数量。
**输出** 本层计数、地块预览和独立地图。


In [ ]:
screen_stage(3, "10_filter_food",
             "Threshold: median across the initial plots. Source: NYS Retail Food Stores.")


## 11｜第 4 层：行人伤亡事故
**输入** 上一层保留的地块与660 ft 内的行人伤亡事故起数。
**输出** 本层计数、地块预览和独立地图。


In [ ]:
screen_stage(4, "11_filter_pedestrian",
             "Threshold: median across the initial plots. Source: NYPD crashes, 2021–2025.")


## 12｜第 5 层：空地或地面停车场
**输入** 上一层保留的地块与地块现状用途编码。
**输出** 本层计数、地块预览和独立地图。图上标出空地、地面停车场与候选编号。


In [ ]:
screen_stage(5, "12_filter_space_type",
             "Source: MapPLUTO. Site use, ownership and access require field verification.")
